In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [5]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
utkface-new.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
import zipfile
zip = zipfile.ZipFile("/content/utkface-new.zip",'r')
zip.extractall("/content")
zip.close()

In [9]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [10]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [47]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [48]:
len(age)

23708

In [49]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [50]:
df.shape

(23708, 3)

In [51]:
df.head()

,age,gender,img
0,13,0,13_0_0_20170110225302179.jpg.chip.jpg
1,54,0,54_0_3_20170119183920621.jpg.chip.jpg
2,2,0,2_0_4_20161221192937540.jpg.chip.jpg
3,35,0,35_0_2_20170116173616956.jpg.chip.jpg
4,26,0,26_0_4_20170117195459004.jpg.chip.jpg


In [52]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [53]:
train_df.shape

(20000, 3)

In [54]:
test_df.shape

(3708, 3)

In [99]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [138]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='raw')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='raw')


def generator_wrapper(gen):
    for x, y in gen:
        yield x, {'age': y[:, 0], 'gender': y[:, 1]}


train_gen = generator_wrapper(train_generator)
test_gen = generator_wrapper(test_generator)

x, y = next(train_gen)

print("X shape:", x.shape)
print("Y values:", y)

Found 20000 validated image filenames.
Found 3708 validated image filenames.
X shape: (32, 200, 200, 3)
Y values: {'age': array([34, 67, 26,  8, 30, 43,  3, 12, 22, 26, 72, 52, 31, 53, 65, 35, 35,
       56, 90, 35, 54, 62, 21, 17, 26, 87, 81, 68, 32, 29, 51,  1]), 'gender': array([1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0,
       1, 1, 0, 1, 1, 0, 1, 1, 0, 0])}


In [139]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [140]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [147]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [148]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [149]:
from tensorflow.keras.utils import plot_model

In [150]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [151]:
model.fit(train_gen, batch_size=32, epochs=10, validation_data=test_gen)

Epoch 1/10
      1/Unknown 25s 25s/step - age_loss: 35.1314 - age_mae: 35.1314 - gender_accuracy: 0.2812 - gender_loss: 0.8137 - loss: 115.6889

KeyboardInterrupt: 